# Team 10 · Seed 15 — train YOLO26n-seg trên Colab

Chạy `train/train.py`: nạp **nguyên văn** các ô thư viện của notebook BTC (`train_and_export.ipynb`), chạy Bước 0→6 (kiểm tra dữ liệu, xuất ONNX, tự chấm như hệ thống, `run_report.json`). Chỉ Bước 3 được tùy biến: train trên danh sách có lặp ảnh lớp hiếm (RFS) + bộ tham số đã chọn.

Cần trong Google Drive, thư mục `AI20k/`: `du_lieu.zip` và `train_and_export.ipynb`.
Checkpoint lưu thẳng vào `AI20k/runs/<VERSION>/` mỗi epoch → nếu Colab ngắt kết nối, chạy lại ô 4 là **tự train tiếp**.
Kết quả cuối (`best.pt`, `submission.zip`, `run_report.json`, …) ở `AI20k/models/<VERSION>/`.

In [ ]:
#@title 1. GPU + Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Code + thư viện đúng phiên bản BTC
import os
DRIVE_DIR = '/content/drive/MyDrive/AI20k'
if not os.path.isdir('/content/repo'):
    !git clone -q -b data-pipeline https://github.com/nnt7923/Team-10---Seed-15---Hackathon.git /content/repo
else:
    !git -C /content/repo pull -q
!cp "$DRIVE_DIR/train_and_export.ipynb" /content/train_and_export.ipynb
!pip install -q --disable-pip-version-check ultralytics==8.4.214 onnx==1.23.2 onnxslim==0.1.99 onnxruntime==1.32.0 pycocotools psutil
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
#@title 3. Giải nén dữ liệu ra ổ /content (nhanh hơn đọc từ Drive)
import os, time
t = time.time()
if not os.path.exists('/content/data/train.txt'):
    !cp "$DRIVE_DIR/du_lieu.zip" /content/du_lieu.zip
    !unzip -q /content/du_lieu.zip -d /content/data && rm /content/du_lieu.zip
!ls /content/data; wc -l /content/data/train.txt /content/data/val.txt
print(f'{time.time() - t:.0f}s')

In [ ]:
#@title 4. Train (chạy lại ô này sau khi mất kết nối = train tiếp từ last.pt)
VERSION = 'v1'  #@param {type:'string'}
EPOCHS = 30     #@param {type:'integer'}
BATCH = 32      #@param {type:'integer'}
WORKERS = 2     #@param {type:'integer'}
import os
runs = f'{DRIVE_DIR}/runs'
resume = '--resume' if os.path.exists(f'{runs}/{VERSION}/weights/last.pt') else ''
reuse = '--reuse-data' if os.path.exists(f'/content/work/{VERSION}/dataset_state.json') else ''
!cd /content/repo && BTC_NOTEBOOK=/content/train_and_export.ipynb PYTHONUNBUFFERED=1 python train/train.py --version $VERSION --data /content/data --work /content/work --epochs $EPOCHS --batch $BATCH --workers $WORKERS --project "$runs" --save-dir "$DRIVE_DIR/models" $resume $reuse 2>&1 | tee -a "$DRIVE_DIR/train_$VERSION.log" | grep -v -E '^\s+[0-9]+/[0-9]+ .*it/s' 

In [ ]:
#@title 5. Kết quả
!ls -la "$DRIVE_DIR/models/$VERSION"
!cat "$DRIVE_DIR/models/$VERSION/val_selfcheck.json"